# Realistic LLM Use Case: When LLMs Actually Win
**SNIA DSN AI Stack Webinar Series — 2026**

This notebook demonstrates a realistic storage engineering task — classifying **unstructured error log messages** into 6 categories — where LLM fine-tuning meaningfully outperforms traditional ML.

## Two Modes

| Mode | What happens | Time |
|------|-------------|------|
| **Full Training** (Step 8A) | Train LoRA adapter from scratch on GPU | ~20 min on T4 |
| **Demo** (Step 8B) | Load pre-trained adapter from HuggingFace Hub | ~2 min |

## Prerequisites
- GPU runtime (T4 or A100) — Go to **Runtime > Change runtime type > T4 GPU**
- No API keys needed — everything runs locally

## How to Use
1. Run Steps 1-7 (data generation and baseline)
2. Choose **either** Step 8A (full training) **or** Step 8B (demo mode) — not both
3. Run Steps 9-12 (evaluation and analysis)

## Step 1: Install Dependencies

Install all required packages, check GPU availability, and record the start time.

In [ ]:
# Step 1: Install Dependencies
!pip install -q torch transformers datasets accelerate peft trl scikit-learn xgboost matplotlib seaborn sentencepiece

import torch
import time as _time

_notebook_start = _time.time()

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Memory: {mem:.1f} GB")
else:
    print("WARNING: GPU recommended for SFT training.")
    print("Go to Runtime > Change runtime type > T4 GPU")

## Step 2: Define Error Categories

Six categories of storage infrastructure errors, each with multiple template variations. These templates create realistic, diverse log messages that challenge traditional ML approaches.

| Category | What it covers |
|----------|---------------|
| Drive Failure | SMART failures, media errors, timeouts, RAID degradation |
| Network Issue | CRC errors, iSCSI timeouts, NFS latency, FC port flaps |
| Capacity Warning | Pool usage, inode limits, quota exceeded, dedup decline |
| Performance Degradation | Latency spikes, throughput drops, IOPS throttling |
| Configuration Error | RAID misconfig, replication lag, multipath errors |
| Firmware Bug | Known defects, memory leaks, race conditions, cache bugs |

In [ ]:
# Step 2: Define Error Categories
import random
import numpy as np
from collections import Counter

random.seed(42)
np.random.seed(42)

ERROR_CATEGORIES = {
    "Drive Failure": {
        "templates": [
            "Critical: Drive {dev} in bay {bay} reported {count} uncorrectable read errors in the last {hours} hours. SMART status changed to FAILING. Predictive failure analysis recommends immediate replacement.",
            "Alert: SSD {dev} serial {serial} has exceeded wear level threshold ({pct}% endurance used). Write amplification factor is {waf}x. Drive relocated to read-only mode pending replacement.",
            "Emergency: NVMe drive {dev} disappeared from PCIe bus after reporting {count} media errors. Controller logs show thermal throttling events preceding the failure. Enclosure slot {bay} requires inspection.",
            "Warning: Drive {dev} experiencing intermittent timeout errors (avg response {ms}ms, expected <10ms). Grown defect list increased by {count} entries this week. Recommend proactive replacement before complete failure.",
            "Disk {dev} in RAID group {rg} marked as degraded after {count} consecutive I/O errors. Automatic rebuild initiated on hot spare. ETA: {hours} hours at current rebuild rate.",
        ],
        "vars": {
            "dev": ["sd{}", "nvme{}n1", "pd{}"],
            "bay": lambda: random.randint(0, 47),
            "count": lambda: random.randint(3, 500),
            "hours": lambda: random.randint(1, 72),
            "serial": lambda: ''.join(random.choices('ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789', k=10)),
            "pct": lambda: random.randint(85, 99),
            "waf": lambda: round(random.uniform(1.5, 8.0), 1),
            "ms": lambda: random.randint(50, 5000),
            "rg": lambda: f"rg{random.randint(0, 15)}",
        }
    },
    "Network Issue": {
        "templates": [
            "Storage network port {port} on node {node} detected {count} CRC errors in {mins} minutes. Link speed auto-negotiated down from {speed_from}Gbps to {speed_to}Gbps. Suspect faulty SFP or cable.",
            "iSCSI target {target} unreachable from initiator {initiator}. TCP retransmissions exceeded threshold ({count} retransmits in {secs}s). Multipath failover triggered to secondary path.",
            "NFS exports from {server} experiencing elevated latency ({ms}ms avg, baseline {baseline}ms). Network trace shows {count} packet drops at switch {switch}. MTU mismatch suspected between {mtu1} and {mtu2} byte frames.",
            "Fibre Channel port {port} logged out {count} times in the last hour. RSCN storm detected — fabric instability affecting zone {zone}. Recommend isolating problematic HBA.",
            "RDMA connection between {node1} and {node2} failed with error {errno}. InfiniBand subnet manager reports {count} link flaps on switch port {port}. Network team investigating physical layer.",
        ],
        "vars": {
            "port": lambda: f"eth{random.randint(0,7)}" if random.random() > 0.5 else f"fc{random.randint(0,3)}",
            "node": lambda: f"node-{random.randint(1, 32):02d}",
            "node1": lambda: f"node-{random.randint(1, 16):02d}",
            "node2": lambda: f"node-{random.randint(17, 32):02d}",
            "count": lambda: random.randint(5, 1000),
            "mins": lambda: random.randint(1, 60),
            "secs": lambda: random.randint(5, 300),
            "speed_from": lambda: random.choice([100, 50, 25]),
            "speed_to": lambda: random.choice([10, 25, 1]),
            "target": lambda: f"iqn.2024-01.com.storage:target{random.randint(1,20)}",
            "initiator": lambda: f"iqn.2024-01.com.host:init{random.randint(1,50)}",
            "server": lambda: f"nfs-{random.randint(1, 8):02d}.internal",
            "ms": lambda: random.randint(20, 500),
            "baseline": lambda: random.randint(1, 5),
            "switch": lambda: f"sw-{random.choice(['core','tor','leaf'])}-{random.randint(1,12)}",
            "mtu1": lambda: random.choice([1500, 9000]),
            "mtu2": lambda: random.choice([1500, 9000]),
            "zone": lambda: f"zone_{random.choice(['prod','backup','repl'])}_{random.randint(1,10)}",
            "errno": lambda: f"0x{random.randint(0x1000, 0xFFFF):04X}",
        }
    },
    "Capacity Warning": {
        "templates": [
            "Storage pool {pool} usage at {pct}% ({used}TB of {total}TB). At current growth rate of {growth}GB/day, pool will reach 95% in {days} days. Thin provisioning overcommit ratio: {ratio}x.",
            "Volume {vol} on aggregate {aggr} has exceeded {pct}% capacity threshold. Snapshot reserve ({snap_pct}% = {snap_tb}TB) consuming more than expected. {count} snapshots older than {age} days should be reviewed.",
            "Filesystem {fs} approaching inode limit: {inodes}M of {max_inodes}M inodes used ({pct}%). Small file workload detected — average file size {avg_kb}KB. Consider increasing inode density or migrating to a filesystem with dynamic inode allocation.",
            "Quota exceeded for project {project}: {used}TB used of {quota}TB allocation. Top consumers: {user1} ({u1_tb}TB), {user2} ({u2_tb}TB). Automated notification sent to project owner.",
            "Deduplication savings declining on pool {pool}: current ratio {ratio}x (was {old_ratio}x last month). Unique data ingestion rate suggests workload change. Estimated {days} days until pool reaches warning threshold at {growth}TB/week growth.",
        ],
        "vars": {
            "pool": lambda: f"pool_{random.choice(['ssd','hdd','nvme','hybrid'])}_{random.randint(1,8)}",
            "pct": lambda: random.randint(80, 98),
            "used": lambda: random.randint(50, 900),
            "total": lambda: random.randint(100, 1000),
            "growth": lambda: random.randint(10, 500),
            "days": lambda: random.randint(3, 90),
            "ratio": lambda: round(random.uniform(1.2, 4.0), 1),
            "old_ratio": lambda: round(random.uniform(2.0, 6.0), 1),
            "vol": lambda: f"vol_{random.choice(['db','app','logs','media'])}_{random.randint(1,20)}",
            "aggr": lambda: f"aggr{random.randint(1, 8)}",
            "snap_pct": lambda: random.randint(10, 30),
            "snap_tb": lambda: round(random.uniform(1, 50), 1),
            "count": lambda: random.randint(10, 200),
            "age": lambda: random.randint(30, 365),
            "fs": lambda: f"/data/{random.choice(['research','engineering','analytics'])}",
            "inodes": lambda: round(random.uniform(50, 250), 1),
            "max_inodes": lambda: 256,
            "avg_kb": lambda: round(random.uniform(0.5, 16), 1),
            "project": lambda: f"proj-{random.choice(['genomics','cfd','ml-training','rendering'])}",
            "quota": lambda: random.randint(10, 200),
            "user1": lambda: random.choice(["jsmith", "akumar", "mchen", "lgarcia"]),
            "user2": lambda: random.choice(["bwilson", "tlee", "rpatel", "kjones"]),
            "u1_tb": lambda: round(random.uniform(5, 80), 1),
            "u2_tb": lambda: round(random.uniform(3, 50), 1),
        }
    },
    "Performance Degradation": {
        "templates": [
            "Latency spike detected on volume {vol}: p99 latency jumped from {baseline}ms to {current}ms over the last {mins} minutes. Queue depth averaging {qd} (normal: {normal_qd}). Possible noisy neighbor on shared controller {ctrl}.",
            "Throughput drop on pool {pool}: sustained {current}MB/s vs baseline {baseline}MB/s. IOstat shows {pct}% disk utilization with {await}ms average wait time. Cache hit rate declined from {cache_baseline}% to {cache_current}%.",
            "Read latency regression after firmware update on controller {ctrl}: average read latency {current}us (pre-update baseline: {baseline}us). Affects {count} volumes across {node_count} nodes. Vendor case {case} opened.",
            "IOPS throttling active on QoS policy {policy}: requested {requested}K IOPS, delivering {delivered}K IOPS. {count} workloads competing for {max}K IOPS ceiling on shared tier. Priority adjustment recommended.",
            "Metadata operation slowdown on filesystem {fs}: directory listing operations taking {current}s (expected <{expected}s). Directory {dir} contains {files} files. Possible B-tree fragmentation or journal contention.",
        ],
        "vars": {
            "vol": lambda: f"vol_{random.choice(['prod','staging','dev'])}_{random.randint(1,30)}",
            "baseline": lambda: random.randint(1, 10),
            "current": lambda: random.randint(50, 500),
            "mins": lambda: random.randint(5, 120),
            "qd": lambda: random.randint(64, 512),
            "normal_qd": lambda: random.randint(8, 32),
            "ctrl": lambda: f"ctrl-{random.choice(['a','b'])}-{random.randint(1,4)}",
            "pool": lambda: f"pool_{random.choice(['flash','hybrid'])}_{random.randint(1,4)}",
            "pct": lambda: random.randint(85, 100),
            "await": lambda: round(random.uniform(10, 200), 1),
            "cache_baseline": lambda: random.randint(80, 98),
            "cache_current": lambda: random.randint(20, 60),
            "count": lambda: random.randint(5, 50),
            "node_count": lambda: random.randint(2, 16),
            "case": lambda: f"SR-{random.randint(100000, 999999)}",
            "policy": lambda: f"qos_{random.choice(['gold','silver','bronze'])}",
            "requested": lambda: random.randint(10, 200),
            "delivered": lambda: random.randint(5, 50),
            "max": lambda: random.randint(100, 500),
            "fs": lambda: f"/mnt/{random.choice(['data','scratch','home'])}",
            "expected": lambda: round(random.uniform(0.1, 1.0), 1),
            "dir": lambda: f"/data/{random.choice(['inbox','logs','temp','exports'])}/",
            "files": lambda: f"{random.randint(100, 10000)}K",
        }
    },
    "Configuration Error": {
        "templates": [
            "RAID rebuild failed on group {rg}: hot spare {dev} incompatible — {spare_size}TB drive cannot replace {orig_size}TB member. Auto-rebuild policy requires matching or larger capacity. Manual intervention required.",
            "Replication lag alert: async mirror from {src} to {dst} has {lag} hours of unacknowledged data ({pending}GB pending). RPO SLA ({rpo}h) violated. Check WAN bandwidth — current utilization {wan_pct}%.",
            "Snapshot schedule {schedule} on volume {vol} creating {count} snapshots/day instead of configured {expected}/day. Cron expression '{cron}' evaluating incorrectly after timezone change. Excess snapshots consuming {extra_tb}TB.",
            "Multipath configuration error on host {host}: path {path_a} and {path_b} both active but different ALUA target port groups. I/O may route through non-optimized path, causing {penalty}x latency penalty.",
            "Tiering policy misconfiguration on pool {pool}: {pct}% of hot data residing on {slow_tier} tier instead of {fast_tier}. Promotion threshold set to {thresh}% — should be {correct_thresh}% for this workload pattern. Estimated performance impact: {impact}% latency increase.",
        ],
        "vars": {
            "rg": lambda: f"rg{random.randint(0, 15)}",
            "dev": lambda: f"sd{chr(random.randint(97, 122))}",
            "spare_size": lambda: random.choice([1.2, 2.4, 4.0]),
            "orig_size": lambda: random.choice([4.0, 8.0, 16.0]),
            "src": lambda: f"site-{random.choice(['east','west','central'])}",
            "dst": lambda: f"site-{random.choice(['dr','backup','archive'])}",
            "lag": lambda: round(random.uniform(2, 48), 1),
            "pending": lambda: random.randint(100, 5000),
            "rpo": lambda: random.choice([1, 4, 8]),
            "wan_pct": lambda: random.randint(85, 100),
            "schedule": lambda: f"snap_{random.choice(['hourly','daily','weekly'])}",
            "vol": lambda: f"vol_{random.choice(['prod','db','app'])}_{random.randint(1,10)}",
            "count": lambda: random.randint(24, 288),
            "expected": lambda: random.choice([4, 12, 24]),
            "cron": lambda: random.choice(["0 */1 * * *", "*/15 * * * *", "0 0 * * *"]),
            "extra_tb": lambda: round(random.uniform(0.5, 20), 1),
            "host": lambda: f"esxi-{random.randint(1,20):02d}",
            "path_a": lambda: f"/dev/mapper/mpath{chr(random.randint(97, 104))}",
            "path_b": lambda: f"/dev/mapper/mpath{chr(random.randint(105, 112))}",
            "penalty": lambda: round(random.uniform(2, 10), 1),
            "pool": lambda: f"pool_tier_{random.randint(1,4)}",
            "pct": lambda: random.randint(40, 80),
            "slow_tier": lambda: random.choice(["capacity-HDD", "archive-cold"]),
            "fast_tier": lambda: random.choice(["performance-SSD", "extreme-NVMe"]),
            "thresh": lambda: random.randint(5, 20),
            "correct_thresh": lambda: random.randint(40, 70),
            "impact": lambda: random.randint(100, 500),
        }
    },
    "Firmware Bug": {
        "templates": [
            "Known firmware defect FW-{fw_id}: Controller {ctrl} running version {ver} may corrupt metadata during concurrent snapshot and clone operations. Vendor advisory {adv} recommends upgrade to {fix_ver}. {count} affected volumes identified.",
            "NVMe drive firmware {ver} on {dev} exhibits write amplification bug: reported WA factor {reported}x vs actual {actual}x. Endurance calculations unreliable — true remaining life may be {life_pct}% lower than reported. Affects {model} drives manufactured before {date}.",
            "Storage OS version {ver} memory leak in dedup engine: resident memory growing {rate}MB/hour. System will OOM in approximately {hours} hours at current rate. Workaround: restart dedup service every {restart}h. Fix expected in version {fix_ver}.",
            "Race condition in replication module (firmware {ver}): under high load ({iops}K+ IOPS), consistency group snapshots may include uncommitted writes. Data integrity risk for {count} replicated volumes. Emergency patch {patch} available.",
            "Cache coherency bug in dual-controller firmware {ver}: after controller failover, {pct}% of dirty cache pages on failed controller not properly destaged. Potential for {count} blocks of data loss per event. Affects {model} systems with >{mem}GB cache.",
        ],
        "vars": {
            "fw_id": lambda: f"{random.randint(2024,2026)}-{random.randint(1,999):03d}",
            "ctrl": lambda: f"ctrl-{random.choice(['a','b'])}",
            "ver": lambda: f"{random.randint(3,8)}.{random.randint(0,9)}.{random.randint(0,9)}",
            "adv": lambda: f"SA-{random.randint(2024,2026)}-{random.randint(1,200):03d}",
            "fix_ver": lambda: f"{random.randint(8,12)}.{random.randint(0,9)}.{random.randint(1,9)}",
            "count": lambda: random.randint(2, 100),
            "dev": lambda: f"nvme{random.randint(0,15)}n1",
            "reported": lambda: round(random.uniform(1.0, 2.0), 1),
            "actual": lambda: round(random.uniform(3.0, 8.0), 1),
            "life_pct": lambda: random.randint(20, 60),
            "model": lambda: random.choice(["FlashArray//X70", "NetApp AFF A400", "PowerStore 9200T", "Unity XT 880F"]),
            "date": lambda: f"2025-{random.randint(1,12):02d}",
            "rate": lambda: round(random.uniform(10, 200), 1),
            "hours": lambda: random.randint(12, 168),
            "restart": lambda: random.choice([6, 12, 24]),
            "iops": lambda: random.randint(50, 500),
            "patch": lambda: f"HP-{random.randint(10000,99999)}",
            "pct": lambda: round(random.uniform(0.1, 5.0), 1),
            "mem": lambda: random.choice([64, 128, 256]),
        }
    },
}

CATEGORIES = list(ERROR_CATEGORIES.keys())
print(f"Error categories: {CATEGORIES}")
print(f"Templates per category: {[len(v['templates']) for v in ERROR_CATEGORIES.values()]}")

## Step 3: Generate Synthetic Error Log Dataset

Generate 600 log messages (100 per category) by filling templates with randomized variable values. Each message is realistic enough to challenge a classifier.

In [ ]:
# Step 3: Generate Synthetic Error Log Dataset
import re

def fill_template(template, var_defs):
    """Fill a template with random variable values."""
    filled = template
    # Collect all needed variables
    import re
    needed = set(re.findall(r'\{(\w+)\}', template))
    values = {}
    for var_name in needed:
        if var_name in var_defs:
            v = var_defs[var_name]
            if callable(v):
                values[var_name] = v()
            elif isinstance(v, list):
                choice = random.choice(v)
                if '{}' in choice:
                    choice = choice.format(chr(random.randint(97, 122)))
                values[var_name] = choice
            else:
                values[var_name] = v
        else:
            values[var_name] = f"UNKNOWN_{var_name}"
    return filled.format(**values)

def generate_log_message(category):
    """Generate a single log message for a given category."""
    cat_def = ERROR_CATEGORIES[category]
    template = random.choice(cat_def["templates"])
    return fill_template(template, cat_def["vars"])

# Generate 600 samples (100 per category)
samples = []
for cat in CATEGORIES:
    for _ in range(100):
        msg = generate_log_message(cat)
        samples.append({"text": msg, "label": cat})

random.shuffle(samples)
print(f"Generated {len(samples)} log messages")
print(f"Categories: {dict(Counter(s['label'] for s in samples))}")

## Step 4: Explore the Data

Display 2 examples per category to see the variety of phrasing. Notice how the same error type can be expressed with completely different vocabulary.

In [ ]:
# Step 4: Explore the Data
for cat in CATEGORIES:
    cat_samples = [s for s in samples if s['label'] == cat][:2]
    print(f"\n{'='*70}")
    print(f"  {cat}")
    print(f"{'='*70}")
    for i, s in enumerate(cat_samples):
        print(f"\n  Example {i+1}:")
        # Word-wrap at ~70 chars
        words = s['text'].split()
        line = "    "
        for w in words:
            if len(line) + len(w) + 1 > 74:
                print(line)
                line = "    " + w
            else:
                line += " " + w if line.strip() else "    " + w
        if line.strip():
            print(line)
print()

## Step 5: Baseline — TF-IDF + XGBoost

First, we establish a traditional ML baseline. TF-IDF converts text to word-frequency features, then XGBoost classifies. This is a strong baseline for text classification — but it treats each message as a bag of words, ignoring word order and context.

In [ ]:
# Step 5: Baseline — TF-IDF + XGBoost
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
from xgboost import XGBClassifier
import time

# Prepare data
texts = [s['text'] for s in samples]
labels = [s['label'] for s in samples]

le = LabelEncoder()
y = le.fit_transform(labels)

# Train/test split
texts_train, texts_test, y_train, y_test = train_test_split(
    texts, y, test_size=0.2, random_state=42, stratify=y
)

# TF-IDF vectorization
print("Vectorizing with TF-IDF...")
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english')
X_train_tfidf = tfidf.fit_transform(texts_train)
X_test_tfidf = tfidf.transform(texts_test)

print(f"Vocabulary size: {len(tfidf.vocabulary_)}")
print(f"Feature matrix: {X_train_tfidf.shape}")

# Train XGBoost
print("\nTraining XGBoost on TF-IDF features...")
t0 = time.time()
xgb_text = XGBClassifier(n_estimators=200, max_depth=6, random_state=42,
                          use_label_encoder=False, eval_metric='mlogloss')
xgb_text.fit(X_train_tfidf, y_train)
xgb_train_time = time.time() - t0

# Evaluate
xgb_preds = xgb_text.predict(X_test_tfidf)
xgb_accuracy = accuracy_score(y_test, xgb_preds)

print(f"\nTraining time: {xgb_train_time:.2f}s")
print(f"Accuracy: {xgb_accuracy:.1%}")
print(f"\nClassification Report:")
print(classification_report(y_test, xgb_preds, target_names=le.classes_))

## Step 6: Why XGBoost Struggles Here

The accuracy above is decent but limited. Three common failure modes:

1. **Vocabulary overlap:** "latency" appears in multiple categories — TF-IDF cannot distinguish "latency spike" (Performance) from "latency penalty" (Configuration Error)
2. **Novel phrasing:** Templates generate diverse phrasings — "disappeared from PCIe bus" and "marked as degraded" both mean drive failure, but share no n-grams
3. **Contextual dependencies:** "firmware version" is neutral, but combined with "race condition" it signals a firmware bug — TF-IDF sees them independently

An LLM processes the full message as a sequence, understanding context and meaning rather than just word frequency.

## Step 7: Prepare SFT Training Data

Format each sample as a prompt/completion pair for supervised fine-tuning. The prompt includes the full log message and asks for a classification; the completion is the category label. This is the same format used in the Post-Training Pipeline.

In [ ]:
# Step 7: Prepare SFT Training Data
from datasets import Dataset
from sklearn.model_selection import train_test_split as tts2

def format_for_sft(sample):
    prompt = (
        f"Classify the following storage error log message into one of these categories: "
        f"{', '.join(CATEGORIES)}.\n\n"
        f"Log message: {sample['text']}\n\n"
        f"Classification:"
    )
    completion = f" {sample['label']}"
    return {"prompt": prompt, "completion": completion}

all_formatted = [format_for_sft(s) for s in samples]

# Split formatted data the same way as Step 5
random.seed(42)
np.random.seed(42)
fmt_train, fmt_test = tts2(all_formatted, test_size=0.2, random_state=42,
                            stratify=[s['label'] for s in samples])

train_dataset = Dataset.from_dict({
    "prompt": [s["prompt"] for s in fmt_train],
    "completion": [s["completion"] for s in fmt_train],
})

print(f"SFT training samples: {len(train_dataset)}")
print(f"\nExample training input:")
print(f"  Prompt (first 200 chars): {fmt_train[0]['prompt'][:200]}...")
print(f"  Completion: {fmt_train[0]['completion']}")

## Step 8A: Full Training Mode (skip for demo)

Train a LoRA adapter on SmolLM2-360M from scratch. This takes ~20 minutes on a T4 GPU. After training, the adapter is saved locally and can optionally be uploaded to the HuggingFace Hub.

**Skip this cell** if you want a quick demo — use Step 8B instead to load a pre-trained adapter.

In [ ]:
# Step 8A: Full Training Mode (~20 min on T4)
# SKIP THIS CELL for demo mode — use Step 8B instead

from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

set_seed(42)

MODEL_NAME = "HuggingFaceTB/SmolLM2-360M"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16, device_map="auto"
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# LoRA configuration — same settings as Post-Training Pipeline
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

# Training configuration
training_args = SFTConfig(
    output_dir="/tmp/sft_error_logs",
    num_train_epochs=4,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    warmup_ratio=0.1,
    logging_steps=10,
    save_strategy="no",
    bf16=torch.cuda.is_available(),
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

print("\nStarting SFT training...")
t0 = time.time()
trainer.train()
sft_train_time = time.time() - t0
print(f"\nTraining completed in {sft_train_time/60:.1f} minutes")

# Save adapter locally
model.save_pretrained("/tmp/sft_error_logs/final_adapter")
tokenizer.save_pretrained("/tmp/sft_error_logs/final_adapter")
print("Adapter saved to /tmp/sft_error_logs/final_adapter")

# Optional: upload to HuggingFace Hub (uncomment and set your repo)
# model.push_to_hub("your-username/smollm2-error-classifier-sft")
# tokenizer.push_to_hub("your-username/smollm2-error-classifier-sft")
# print("Adapter uploaded to HuggingFace Hub")

## Step 8B: Demo Mode — Load Pre-trained Adapter

Load a pre-trained LoRA adapter from the HuggingFace Hub. This skips the ~20 minute training step and lets you jump straight to evaluation. The adapter was trained with the exact same configuration as Step 8A.

In [ ]:
# Step 8B: Demo Mode — Load Pre-trained Adapter from HuggingFace Hub
# USE THIS CELL instead of Step 8A for a quick demo

from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from peft import PeftModel

set_seed(42)

MODEL_NAME = "HuggingFaceTB/SmolLM2-360M"
ADAPTER_REPO = "provandal/smollm2-error-classifier-sft"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading base model: {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16, device_map="auto"
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loading pre-trained adapter: {ADAPTER_REPO}...")
model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)
print("Adapter loaded successfully.")

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Parameters: {trainable:,} trainable / {total:,} total")

# Mark that we skipped training
sft_train_time = 0

## Step 9: Evaluate the SFT Model

Run inference on the test set and compare predictions against ground truth. Each sample is formatted with the same prompt template used during training.

In [ ]:
# Step 9: Evaluate the SFT Model
model.eval()

# Reconstruct test set
_, test_samples_eval = tts2(samples, test_size=0.2, random_state=42,
                             stratify=[s['label'] for s in samples])

sft_preds = []
sft_correct = 0
total_test = len(test_samples_eval)

print(f"Evaluating SFT model on {total_test} test samples...")
for idx, sample in enumerate(test_samples_eval):
    if (idx + 1) % 20 == 0 or idx == 0:
        print(f"  Progress: {idx+1}/{total_test}")

    prompt = (
        f"Classify the following storage error log message into one of these categories: "
        f"{', '.join(CATEGORIES)}.\n\n"
        f"Log message: {sample['text']}\n\n"
        f"Classification:"
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs, max_new_tokens=20, do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    generated = tokenizer.decode(output_ids[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

    # Check if any category name appears in the output
    predicted = None
    for cat in CATEGORIES:
        if cat.lower() in generated.lower():
            predicted = cat
            break

    is_correct = predicted == sample['label']
    if is_correct:
        sft_correct += 1
    sft_preds.append({
        'true': sample['label'],
        'predicted': predicted or generated[:50],
        'raw_output': generated[:100],
        'correct': is_correct,
    })

sft_accuracy = sft_correct / len(test_samples_eval)
print(f"\nSFT Accuracy: {sft_accuracy:.1%} ({sft_correct}/{len(test_samples_eval)})")

# Per-category breakdown
print(f"\nPer-Category Results:")
print("-" * 50)
for cat in CATEGORIES:
    cat_preds = [p for p in sft_preds if p['true'] == cat]
    cat_correct = sum(1 for p in cat_preds if p['correct'])
    cat_total = len(cat_preds)
    pct = cat_correct / cat_total if cat_total > 0 else 0
    bar = "#" * int(pct * 10) + "." * (10 - int(pct * 10))
    print(f"  {cat:<25s} {bar} {cat_correct}/{cat_total} ({pct:.0%})")

## Step 10: Head-to-Head Comparison

Side-by-side charts comparing TF-IDF + XGBoost vs SFT fine-tuning on overall accuracy and per-category accuracy, plus a summary table with training time, model size, and qualitative differences.

In [ ]:
# Step 10: Head-to-Head Comparison
import matplotlib.pyplot as plt
import pandas as pd

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Accuracy comparison
models = ['TF-IDF + XGBoost', 'SFT (SmolLM2-360M)']
accs = [xgb_accuracy, sft_accuracy]
colors = ['#f97316', '#3b82f6']
bars = ax1.bar(models, accs, color=colors, alpha=0.8, width=0.5)
ax1.set_ylabel('Accuracy')
ax1.set_title('Text Classification Accuracy')
ax1.set_ylim(0, 1.1)
for bar, val in zip(bars, accs):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{val:.1%}', ha='center', fontweight='bold', fontsize=13)

# Right: Per-category comparison
xgb_cat_acc = {}
sft_cat_acc = {}
for cat in CATEGORIES:
    # XGBoost
    cat_mask = [le.classes_[y_test[i]] == cat for i in range(len(y_test))]
    cat_indices = [i for i, m in enumerate(cat_mask) if m]
    if cat_indices:
        xgb_cat_acc[cat] = sum(1 for i in cat_indices if xgb_preds[i] == y_test[i]) / len(cat_indices)
    # SFT
    cat_sft = [p for p in sft_preds if p['true'] == cat]
    if cat_sft:
        sft_cat_acc[cat] = sum(1 for p in cat_sft if p['correct']) / len(cat_sft)

x = np.arange(len(CATEGORIES))
width = 0.35
ax2.bar(x - width/2, [xgb_cat_acc.get(c, 0) for c in CATEGORIES], width,
        label='TF-IDF + XGBoost', color='#f97316', alpha=0.8)
ax2.bar(x + width/2, [sft_cat_acc.get(c, 0) for c in CATEGORIES], width,
        label='SFT', color='#3b82f6', alpha=0.8)
ax2.set_ylabel('Accuracy')
ax2.set_title('Per-Category Accuracy')
ax2.set_xticks(x)
ax2.set_xticklabels(CATEGORIES, rotation=35, ha='right', fontsize=8)
ax2.legend()
ax2.set_ylim(0, 1.15)

plt.tight_layout()
plt.show()

# Summary table
print("\n" + "=" * 70)
print("  COMPARISON: TF-IDF + XGBoost vs SFT Fine-Tuning")
print("=" * 70)
sft_time_str = f'{sft_train_time/60:.1f} min' if sft_train_time > 0 else 'Pre-trained (skipped)'

comparison_data = {
    'Metric': ['Accuracy', 'Training Time', 'Model Size', 'Hardware', 'Handles Novel Phrasing', 'Contextual Understanding'],
    'TF-IDF + XGBoost': [f'{xgb_accuracy:.1%}', f'{xgb_train_time:.1f}s', '~5 MB', 'CPU', 'Limited', 'None (bag of words)'],
    'SFT (SmolLM2-360M)': [f'{sft_accuracy:.1%}', sft_time_str, '~700 MB', 'GPU', 'Good', 'Full sequence modeling'],
}
comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))
print("=" * 70)

## Step 11: Error Analysis

Examine cases where the SFT model succeeds and TF-IDF + XGBoost fails. This highlights the LLM's advantage in understanding contextual meaning.

In [ ]:
# Step 11: Error Analysis — Where does the LLM win?
print("Cases where SFT succeeds and TF-IDF + XGBoost fails:")
print("=" * 70)

# Align test samples between the two models
sft_test = test_samples_eval
count = 0
for i, (sample, sft_pred) in enumerate(zip(sft_test, sft_preds)):
    if i >= len(y_test):
        break
    xgb_pred_label = le.classes_[xgb_preds[i]] if i < len(xgb_preds) else None
    xgb_wrong = xgb_pred_label != sample['label']
    sft_right = sft_pred['correct']
    
    if xgb_wrong and sft_right and count < 4:
        count += 1
        print(f"\n  Example {count}:")
        print(f"  True label:    {sample['label']}")
        print(f"  XGBoost pred:  {xgb_pred_label} X")
        print(f"  SFT pred:      {sft_pred['predicted']} OK")
        text = sample['text'][:150]
        print(f"  Log message:   {text}...")

if count == 0:
    print("  (No clear examples in this run -- both models may have similar error patterns)")

print(f"\n\nSummary: SFT's advantage comes from understanding the *meaning* of log messages,")
print(f"not just keyword frequency. This matters most when:")
print(f"  - Different categories share vocabulary (latency, errors, failures)")
print(f"  - The same concept is expressed with different words")
print(f"  - Context determines category (cache in 'performance' vs 'firmware')")

## Step 12: Conclusion — A Framework for Choosing Your Approach

| Characteristic | Use Traditional ML | Use LLM Fine-Tuning |
|---|---|---|
| **Input format** | Structured numbers, tables | Unstructured text, natural language |
| **Feature engineering** | Clear, known features | Features hard to extract manually |
| **Vocabulary** | N/A or controlled | Variable, natural language |
| **Context dependency** | Low — features are independent | High — meaning depends on context |
| **Training resources** | CPU, seconds-minutes | GPU, minutes-hours |
| **Model size** | KBs-MBs | Hundreds of MBs |
| **Interpretability** | High (feature importance) | Low (black box) |

**The two notebooks together give you a decision framework:**
- **Structured numeric data** (I/O metrics, sensor readings, performance counters) -> Random Forest / XGBoost
- **Unstructured text** (logs, error messages, incident reports, documentation) -> LLM fine-tuning

**Neither approach is universally better.** The right choice depends on your data, not on what's trending.

Return to the [Post-Training Pipeline](Post_Training_Pipeline.ipynb) to run the full SFT -> DPO -> GRPO pipeline, or see the [Traditional ML Comparison](Traditional_ML_Comparison.ipynb) for the structured data perspective.